# 参考
- https://www.kaggle.com/code/hassan1417/bc2026-claude-tucker-distilled-sed-fork
- コンペへの挑戦を決意してから締め切りまで25日しかなかったため、上記の公開SED学習コードを改修した。

# 試行履歴
- 変更000
  - Googleドライブに接続し、Google Colabで実行可能にする。
- 変更001（コメントアウト）
  - 読み取った音声データから作成したメルスペクトログラムの2次元画像（縦軸 = 周波数、横軸 = 時間）について、時間の情報を残すのではなく周波数の情報を残す。
    - いつ鳴いたかはわからなくなるが、どの音域で鳴いたかがわかるようになるため、種の識別に有利となる。
  - ※スコアが低下したため廃止。
- 変更002（コメントアウト）
  - 画像認識用のニューラルネットワークモデルを、「EfficientNet-B0」から「EfficientNet-V2-S」に変更。
  - ※推論コードが「EfficientNet-B0」を前提としており、スコアが低下したため廃止。
- 変更003
  - 学習データが少ない種を再学習させて、予測精度を上げる。
    - 学習ループを完了させる。
    - 次にBackbone（特徴抽出部分）の重みを固定し、分類ヘッド（SEDヘッド）のみを少数クラスのデータで追加学習する。Backboneを固定することで、多数クラスの学習結果を壊さずに少数クラスの精度だけを向上できる。
- 変更004
  - Model Soupの実装
    - 後半1/3のEpochのチェックポイントを平均化し、Epoch数によるスコアのばらつきを抑える。
- 変更005（コメントアウト）
  - Pseudo labeling （疑似ラベリング）データ追加処理の実装。学習データを増やす。
    - 事前にcsv作成用推論コード（Create_submission_sed_full_csv）で「train_soundscapes」フォルダの全ファイルを推論し、予測スコアを疑似ラベリングデータの元データとして「submission_sed_full.csv」に保存しておく。
    - csvファイルをこのSED学習コードで読み込み、以下のすべてを満たす元データを疑似ラベリングデータとして学習データに追加する。
       - 信頼度が高いウィンドウ（max_logit > 5.0）
       - スコアが0.3を超えるクラス（score_matrix > 0.3）
  - ※変更007に統合したため廃止。
- 変更006（スコアが低下したため廃止）
  - Background noiseの追加。学習時にランダムに背景ノイズを混ぜることで、モデルがノイズに強くなる。
  - ※スコアが低下したため廃止。
- 変更007（スコアが低下したため廃止）
  - ※変更008と併用不可。
  - Pseudo labeling（疑似ラベリング）データ追加処理の実装。学習データを増やす。
    - 事前にcsv作成用推論コード（Create_submission_sed_full_csv）で「train_soundscapes」フォルダの全ファイルを推論し、予測スコアを疑似ラベリングデータの元データとして「submission_sed_full.csv」に保存しておく。
    - 1回目: 推論コード（LB 0.951のアンサンブル）の予測スコアをPseudo labelとして使用。ただし以下のすべてを満たすデータのみを追加する。
      - 信頼度が高いウィンドウ（max_logit > 5.0）
      - べき乗変換でノイズ抑制したスコア（`PSEUDO_POWER`推奨: 1.5〜2.0）
    - 2回目: 1回目で学習したモデルで再推論し、同様のPseudo labelを生成して再学習。
  - ※スコアが低下したため廃止。
- 変更008（スコアが低下したため廃止）
  - ※変更007と併用不可。
  - Self-Distillation（自己蒸留）によるデータ追加処理の実装。学習データを増やす。
    - 事前にcsv作成用推論コード（Create_submission_sed_full_csv）で「train_soundscapes」フォルダの全ファイルを推論し、予測スコアを疑似ラベリングデータの元データとして「submission_sed_full.csv」に保存しておく。
    - 変更007との違い: べき乗変換を使わず、予測スコアをsoft labelとしてそのまま使用する。Teacher modelの予測をStudent modelが蒸留学習することでノイズを抑えながら学習する。
    - 1回目: 推論コード（LB 0.951のアンサンブル）の予測スコアをsoft labelとして使用。ただし以下のすべてを満たすデータのみを追加する。
      - 信頼度が高いウィンドウ（max_logit > 5.0）
      - soft labelのスコア（`DISTILL_ALPHA`でhard labelとの比率を制御、推奨: 0.5）
    - 2回目: 1回目で学習したモデルで再推論し、同様のsoft labelを生成して再学習。
  - ※スコアが低下したため廃止。
- 変更009
  - 2025 1st place・2nd place方式の改善を追加。
    - MixUp blending weightを固定0.5に変更。
    - drop_path_rateを0.1→0.15に変更。
    - WeightedPseudoSamplerを実装（1st・2nd place方式: pseudo labelの信頼度で重み付けサンプリング）。
    - create_pseudo_v2（OOF分離・primary_label_prob > 0.5フィルタ・fold分割）で生成したPseudo labelを使用。

# --- 変更000 削除 ↓↓↓--------------------------------------------------------------
# --- 変更000 削除 ↑↑↑ -------------------------------------------------------------

# --- 変更000 追加 ↓↓↓--------------------------------------------------------------
# --- 変更000 追加 ↑↑↑ -------------------------------------------------------------


# BirdCLEF 2026 — 蒸留SED: 学習 + 推論

**アーキテクチャ:** EfficientNet-B0バックボーン + SEDアテンションヘッド + 凍結Perch v2埋め込み蒸留
**損失:** BCE (0.5 × clip + 0.5 × frame-max) + α · MSE(student_emb, perch_emb)  
**学習データ:** train_audio（焦点録音）+ ラベル付きtrain_soundscapes  
**参考:** https://www.kaggle.com/code/hassan1417/bc2026-claude-tucker-distilled-sed-fork

---

### このノートブックの動作

1. **学習(S6):** Perch蒸留付きEfficientNet-B0 SEDモデルを学習する（5分割交差検証、各分割約25エポック）
2. **少数サンプルの再学習(S7):**
2. **Export** each fold’s best checkpoint to ONNX
3. **推論:** ONNXモデルを使ってテストサウンドスケープを推論する（スコアリング時にPyTorchは不要）

### 必要なKaggleデータセット

| データセット | 内容 | 備考 |
|---|---|---|
| `birdclef-2026` | コンペデータ（train_audio, train_soundscapes, CSV） | 自動添付 |
| `bc2026-waveform-cache` | 事前抽出済み波形キャッシュ（`.pt` int16テンソル） | インプットとして添付 |
| `perch-v2-no-dft-onnx` | `perch_v2_no_dft.onnx` + onnxruntimeホイール | インプットとして添付 |

波形キャッシュには、int16 PyTorchテンソルとして事前に切り出された焦点録音とサウンドスケープウィンドウ、およびメタデータCSVが含まれている。これにより学習中に数千のoggファイルをデコードする必要がなくなる。

### バージョン

V2: コメントと軽微なバグを修正。後処理としてガウシアンスムージングを追加。

## S0 — 環境セットアップ

必要なパッケージをインストールする。onnxruntimeホイールはスコアリング時のネットワーク依存を避けるためKaggleデータセットにバンドルされている。`timm`はImageNet事前学習済み重みのEfficientNetバックボーンを提供する。


In [ ]:
# --- 変更000 追加 ↓↓↓--------------------------------------------------------------

import os
from google.colab import drive
from pathlib import Path

# Googleドライブへの接続と移動
drive.mount('/content/drive', force_remount=True)
WORKING_DIR = "/content/drive/MyDrive/private/Kaggle/BirdCLEF_2026/Your_Work"
os.chdir(WORKING_DIR)
print(f"現在の作業ディレクトリ: {os.getcwd()}")

# -----------------------------------------------------------------
# ノートブックモード
# -----------------------------------------------------------------
NOTEBOOK_MODE = "train"  # "train"（学習） or "infer"（推論）

DEBUG_MODE = False

# -----------------------------------------------------------------
# トレーニングモード
# -----------------------------------------------------------------
TRAINING_MODE = "middle"  # "light" or "middle" or "heavy"
# middle or heavy: FOLDSを[0, 1], [2, 3], [4]に3分割して実行

# -----------------------------------------------------------------
# 出力先パス
# -----------------------------------------------------------------
OUTPUT_DIR = Path(WORKING_DIR + "/training_sed2/output_003_004_009/fold01234_epoch25")
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"OUTPUT_DIR が存在しません: {OUTPUT_DIR}\nos.makedirs(str(OUTPUT_DIR), exist_ok=True) で作成してください。")

# --- 変更000 追加 ↑↑↑ -------------------------------------------------------------


Mounted at /content/drive
現在の作業ディレクトリ: /content/drive/.shortcut-targets-by-id/1EGO2jwqaKnuKe-GvftYfTpVBd5j07uzC/private/Kaggle/BirdCLEF_2026/Your_Work


## S1 — インポートと設定

In [ ]:
# =================================================================
# S1 -- インポートと設定
# =================================================================
import os, sys, time, json, pickle, gc, random, math
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast
import torchaudio
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.special import expit as sigmoid_np
import warnings
warnings.filterwarnings("ignore")

SEED = 42  # 乱数シード: 実験の再現性を確保するために固定する
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU（CUDA）が使えればGPUを、なければCPUを使う
print(f"Device: {device}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name()}")

# ------------------------------------------------------------------
# Notebook Mode
# -----------------------------------------------------------------
MODE = "infer"  # "train": 学習モード / "infer": 推論モード

if MODE == 'train':    #Kaggle struggles with full training pipeline
    DEBUG = True
else:
    DEBUG = False

# -----------------------------------------------------------------
# パス -- Kaggleデータセットのレイアウト
# -----------------------------------------------------------------
COMP_DIR = Path("/kaggle/input/competitions/birdclef-2026")
WAVEFORM_CACHE_DIR = Path("/kaggle/input/datasets/tuckerarrants/birdclef-2026-waveform-cache/waveform_cache")
PERCH_ONNX_PATH = Path("/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/perch_v2_no_dft.onnx")

LABELS_PATH     = COMP_DIR / "train_soundscapes_labels.csv"
TAXONOMY_PATH   = COMP_DIR / "taxonomy.csv"
SAMPLE_SUB_PATH = COMP_DIR / "sample_submission.csv"
TEST_DIR        = COMP_DIR / "test_soundscapes"

OUT_DIR = Path("/kaggle/working")

NUM_CLASSES = 234  # 予測対象の種数（BirdCLEF 2026）
SR = 32000  # サンプリングレート: 1秒あたり32000個のデータ点

# --- 音声長 ---
TRAIN_DURATION = 5    # 学習時の音声の長さ（秒）
VAL_DURATION   = 5    # コンペ評価では常に5秒（変更不可）
TRAIN_SAMPLES  = SR * TRAIN_DURATION
VAL_SAMPLES    = SR * VAL_DURATION

N_FOLDS = 5

# --- メルスペクトログラム ---
N_FFT      = 2048  # フーリエ変換のウィンドウサイズ（大きいほど周波数解像度が高い）
HOP_LENGTH = 512   # フーリエ変換のステップ幅（小さいほど時間解像度が高い）
N_MELS     = 256   # メルスペクトログラムの周波数方向の解像度
FMIN       = 20
FMAX       = 16000

# --- モデル ---
# --- 変更002 削除 ↓↓↓--------------------------------------------------------------
BACKBONE_NAME = "tf_efficientnet_b0.ns_jft_in1k"
# --- 変更002 削除 ↑↑↑ -------------------------------------------------------------
# --- 変更002 追加 ↓↓↓--------------------------------------------------------------
# EfficientNet-B0からEfficientNet-V2-Sに変更する。
# パラメータ数が約4倍になり、特徴抽出能力が向上する。
# 一方でGPUメモリ消費が増えるため、BATCHサイズの調整が必要な場合がある。
# BACKBONE_NAME = "tf_efficientnetv2_s.in21k_ft_in1k"
# --- 変更002 追加 ↑↑↑ -------------------------------------------------------------

# --- Perch蒸留 ---
USE_PERCH_DISTILL = True  # Perch蒸留を使うか（Trueで精度向上するが学習時間が増加）
PERCH_EMBED_DIM   = 1536
ALPHA_DISTILL     = 1.0   # 蒸留損失の重み（大きいほどPerchの特徴に近づける）

# --- 学習 ---
FOLDS  = [0, 1, 2, 3, 4]
EPOCHS = 25
BATCH  = 16 if DEBUG else 64        # ローカルでは64、Kaggleでは16を使用
LR     = 5e-4
MIN_LR = 1e-5
WD     = 1e-4
WARMUP_EPOCHS = 2

# --- アップサンプリング ---
MIN_SAMPLE = 20  # この件数未満の種を「少数サンプル」とみなす（変更003で使用）

# --- オーグメンテーション ---
AUG_PROB = 0.5
AUG_GAIN_DB_RANGE      = (-6.0, 6.0)
AUG_NOISE_SNR_DB_RANGE = (10.0, 30.0)

# --- 変更006 追加 ↓↓↓--------------------------------------------------------------
# --- Background noise ---
# USE_BG_NOISE         = True
# BG_NOISE_PROB        = 0.5
# BG_NOISE_SNR_DB_RANGE_BG = (10, 30)
# BG_NOISE_DIR         = Path(WORKING_DIR + "/input/background_noise")
# --- 変更006 追加 ↑↑↑ -------------------------------------------------------------

# --- MixUp ---
USE_FOCAL_MIXUP    = True
MIXUP_PROB         = 0.5
MIXUP_ALPHA        = 0.4
MIXUP_HARD         = True    # ラベルの和集合（ハード）vs 重み付きブレンド（ソフト）

USE_FOCAL_SC_MIXUP     = True
FOCAL_SC_MIXUP_PROB    = 0.5
FOCAL_SC_MIXUP_ALPHA   = 0.4

# --- FreqMixStyle（デフォルトで無効）---
FREQ_MIXSTYLE_PROB  = 0.0
FREQ_MIXSTYLE_ALPHA = 0.1

# --- SpecAugment ---
FREQ_MASK_PARAM = 10
TIME_MASK_PARAM = 10
NUM_FREQ_MASKS  = 1
NUM_TIME_MASKS  = 2

# --- ソース重み ---
USE_FOCAL           = True
USE_FOCAL_SECONDARY = True
USE_LABELED_SC      = True

# --- 変更007 追加 ↓↓↓--------------------------------------------------------------
# USE_PSEUDO_007     = True
# PSEUDO_LOGIT_MIN   = 5.0
# PSEUDO_POWER       = 1.82   # べき乗変換（1st placeは1.82を使用）
# PSEUDO_SIGMOID_THR = 0.3   # sigmoid後のしきい値
# PSEUDO_SUBM_PATH   = Path(WORKING_DIR + "/input/inference/submission_sed_full.csv")
# SHARES_007         = {"focal": 0.80, "sc": 0.10, "pseudo": 0.10}
# SHARES             = SHARES_007
# SOURCE_WEIGHTS_007 = {"focal": 1.0, "focal_missing": 0.0, "sc": 1.0, "pseudo": 0.5}
# --- 変更007 追加 ↑↑↑ -------------------------------------------------------------

# --- 変更008 追加 ↓↓↓--------------------------------------------------------------
# USE_DISTILL_008    = True
# PSEUDO_LOGIT_MIN = 5.0
# DISTILL_TEMP       = 1.0    # 蒸留温度（1.0でsoft labelそのまま）
# DISTILL_ALPHA      = 0.5    # soft label vs hard labelの比率
# PSEUDO_SUBM_PATH   = Path(WORKING_DIR + "/input/inference/submission_sed_full.csv")
# SHARES_008         = {"focal": 0.80, "sc": 0.10, "pseudo": 0.10}
# SHARES             = SHARES_008
# SOURCE_WEIGHTS_008 = {"focal": 1.0, "focal_missing": 0.0, "sc": 1.0, "pseudo": 0.3}
# --- 変更008 追加 ↑↑↑ -------------------------------------------------------------

# --- 変更009 追加 ↓↓↓--------------------------------------------------------------
USE_PSEUDO_009       = True  # 変更009: OOF分離Pseudo label使用
PSEUDO_LOGIT_MIN_009 = 0.5   # primary_label_prob > 0.5（2nd place方式）
PSEUDO_POWER_009     = 1.82  # べき乗変換（1st place方式）
PSEUDO_SUBM_PATH_009 = Path(WORKING_DIR + "/input/inference/submission_sed_full_filtered.csv")
# ドメインギャップ対策（※Markdown S2を参照）
SHARES_009           = {"focal": 0.80, "sc": 0.10, "pseudo": 0.10}
SHARES               = SHARES_009
# --- 変更009 追加 ↑↑↑ -------------------------------------------------------------

ACTIVE_SOURCES = ["focal", "sc"]
SOURCE_WEIGHTS = {
    "focal":         1.0,
    "focal_missing": 0.0,
    "sc":            1.0,
}

print(f"Backbone: {BACKBONE_NAME}")
print(f"Train duration: {TRAIN_DURATION}s | Mel: {N_MELS} mels, n_fft={N_FFT}, hop={HOP_LENGTH}")
print(f"Distillation: {'ON' if USE_PERCH_DISTILL else 'OFF'} (alpha={ALPHA_DISTILL})")


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Backbone: tf_efficientnet_b0.ns_jft_in1k
Train duration: 5s | Mel: 256 mels, n_fft=2048, hop=512
Distillation: ON (alpha=1.0)


In [ ]:
# --- 変更000 追加 ↓↓↓--------------------------------------------------------------

# 定数の更新
MODE = NOTEBOOK_MODE
DEBUG = DEBUG_MODE
OUTPUT = OUTPUT_DIR

def ts():
    """日本の年月日時分秒を返却する。"""
    JST = timezone(timedelta(hours=9))
    return datetime.now(JST).strftime('%Y-%m-%d %H:%M:%S')

# # バンドルされたホイールからインストール（スコアリング時にネットワーク不要）
# !pip install -q /kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl

from pathlib import Path
import subprocess, sys
from datetime import timezone, timedelta, datetime

# バンドルされたwheelからインストール（スコアリング時にネットワーク不要）
ONNX_WHL = Path(WORKING_DIR + "/wheel/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)],
        check=True,
    )
    print("ONNX Runtime installed")

# --- 変更000 追加 ↑↑↑ -------------------------------------------------------------


ONNX Runtime installed


In [ ]:
if MODE == "train":
    !pip install -q timm torchaudio onnxscript onnx

## S2 — データ読み込み

1. 学習ループ内で同じファイルを何度も読み込むと処理時間が長くなるため、 Google Colab のローカルにファイルをコピー。

2. 以下の3ファイルを読み込む。
- A. 焦点音声メタデータ
  - 教師データ。特定の種だけが鳴いているきれいな音声。
  - 弱ラベル（録音全体に種名が付くだけで、鳴いた時刻は不明）。
- B. サウンドスケープメタデータ
  - 教師データ。色々な種＋雑音が混ざった環境音。
  - 強ラベル（5秒区間ごとに、鳴いている種を明示）。
  - 本番（テスト）に近い実環境音。
- C. 正解ラベル（専門家アノテーション）
  - Bに対する正解データ。

- ※手法
  - ドメインギャップ対策（きれいな単独音 ↔ 本番の雑音まみれの環境音）
    - 混合比
      - 各バッチを (A：B：変更009の疑似ラベル) = (8:1:1) で構成し、最初から本番に近い音に慣らす。
    - 焦点-サウンドスケープMixUp
      - AとBを重ねて「中間の音」を作り、橋渡しする。
    - ※AとBは1つの学習ループの中で順番ではなく同時に実施（Bへの過学習・Aの知識の忘却を防ぐため）。
  - StratifiedKFold（希少種の偏り対策）
    - 各種の比率を保ったまま、どのフォールドにも全種を均等に振り分ける。そうすることで、データ数の少ない希少種が検証用に含まれず検証対象から漏れることを防ぐ。
    - Aに適用。
  - GroupKFold（データリーク対策）
    - 同じ音声データのフォールド分割データは全て学習用、または全て検証用に分類する。そうすることで、検証中に目的の音声以外の環境音で答えを当てて検証スコアが高くなることを防ぐ。
    - Bに適用。
  - 希少種のアップサンプリング
    - 種ごとのデータ数が偏って数件しかない希少種はモデルが学習できないため、その種のデータを複製して水増しする。
    - 単純な複製は過学習となる可能性があるが、オーグメンテーションにより複製するため、実質「少しずつ違う複数バージョン」として学習され効果が出る。
  - S22（録音した場所の識別子）の除外
    - S22はラベルノイズが既知のサイト。
    - 評価からS22を除外し、検証スコアが狂うのを防ぐ。
    - ただし学習には参加させる（S22にしか存在しない種があるため）。

- ※用語
  - フォールド分割
    - 教師データを学習用と検証用に分割する。学習用でモデルを訓練し、検証用で本番データのスコアを見積もる。
  - データリーク
    - 本番では手に入らない情報を使って検証問題を解いてしまい、見積もり段階で実力以上の点が出てしまう状態のこと。
  - オーグメンテーション
    - 学習データをわざと少しずつ変形（音量変化やノイズ追加）して水増しし、丸暗記を防ぎ汎化を上げる手法。
  - ラベルノイズ
    - 正解ラベル自体に誤りが多いこと。

In [ ]:
# --- 変更000 追加 ↓↓↓--------------------------------------------------------------

# =================================================================
# S2.1 -- データのローカルコピー
# =================================================================

import shutil
import concurrent.futures
from datetime import timezone, timedelta

def _copy_one(args):
    """1ファイルをコピーする（並列処理用）"""
    src_path, dst_path = args
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    shutil.copy2(src_path, dst_path)
    return dst_path

# 定数の上書き
COMP_DIR             = Path(WORKING_DIR + "/input/BirdCLEF-2026")
WAVEFORM_CACHE_DIR   = Path(WORKING_DIR + "/input/BirdCLEF-2026-waveform-cache/waveform_cache")
SOUNDSCAPE_CACHE_DIR = Path(WORKING_DIR + "/input/BirdCLEF-2026-waveform-cache/waveform_cache/soundscape")
PERCH_ONNX_PATH      = Path(WORKING_DIR + "/input/perch_v2_no_dft_onnx/perch_v2_no_dft.onnx")
LABELS_PATH     = COMP_DIR / "train_soundscapes_labels.csv"
TAXONOMY_PATH   = COMP_DIR / "taxonomy.csv"
SAMPLE_SUB_PATH = COMP_DIR / "sample_submission.csv"
TEST_DIR        = COMP_DIR / "test_soundscapes"

WAVEFORM_CACHE_LOCAL_DIR = Path("/content/BirdCLEF-2026-waveform-cache/waveform_cache")  # Colabのローカルパス

OUT_DIR = OUTPUT_DIR

if TRAINING_MODE == "light":
    # 10Epoch
    FOLDS  = [0]
    EPOCHS = 10
    BATCH  = 16 if DEBUG else 64  # ローカルでは64、Kaggleでは16を使用
    USE_PERCH_DISTILL = True      # 蒸留あり
elif TRAINING_MODE == "middle":
    # 25Epoch
    FOLDS  = [0, 1, 2, 3, 4]
    EPOCHS = 25
    BATCH  = 16 if DEBUG else 64  # ローカルでは64、Kaggleでは16を使用
    USE_PERCH_DISTILL = True      # 蒸留あり
else:
    # 50Epoch
    FOLDS  = [0, 1, 2, 3, 4]
    EPOCHS = 50
    BATCH  = 16 if DEBUG else 64  # ローカルでは64、Kaggleでは16を使用
    USE_PERCH_DISTILL = True      # 蒸留あり

print(f"Backbone: {BACKBONE_NAME}")
print(f"Train duration: {TRAIN_DURATION}s | Mel: {N_MELS} mels, n_fft={N_FFT}, hop={HOP_LENGTH}")
print(f"Distillation: {'ON' if USE_PERCH_DISTILL else 'OFF'} (alpha={ALPHA_DISTILL})")
print(f"Batch: {BATCH} | Epochs: {EPOCHS} | Folds: {FOLDS}\n")

# WAVEFORM_CACHE_DIRの中身を全てColabローカルに保存して高速化
if os.path.exists(WAVEFORM_CACHE_LOCAL_DIR):
    print(f"{ts()} 波形キャッシュが残っているため、コピーをスキップします。")
else:
    # コピー対象ファイルを全て列挙
    all_files = []
    for root, dirs, files in os.walk(WAVEFORM_CACHE_DIR):
        for f in files:
            src_path = os.path.join(root, f)
            rel_path = os.path.relpath(src_path, WAVEFORM_CACHE_DIR)
            dst_path = os.path.join(WAVEFORM_CACHE_LOCAL_DIR, rel_path)
            all_files.append((src_path, dst_path))

    total = len(all_files)
    print(f"{ts()} コピー開始... 総ファイル数: {total}")

    count = 0
    # 4スレッドで並列コピー（Google Driveのレート制限に引っかかる場合は、max_workersを半減させる）
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        for dst in executor.map(_copy_one, all_files):
            count += 1
            if count % 1000 == 0 or count == total:
                print(f"{ts()} {count}ファイル完了")

    # コピー完了後、読み込み先をローカルに切り替え
    print(f"{ts()} コピー完了")

WAVEFORM_CACHE_DIR = WAVEFORM_CACHE_LOCAL_DIR
# --- 変更000 追加 ↑↑↑ -------------------------------------------------------------

Backbone: tf_efficientnet_b0.ns_jft_in1k
Train duration: 5s | Mel: 256 mels, n_fft=2048, hop=512
Distillation: ON (alpha=1.0)
Batch: 64 | Epochs: 18 | Folds: [0]

2026-06-03 23:41:23 波形キャッシュが残っているため、コピーをスキップします。


In [ ]:
# =================================================================
# S2.2 -- データ読み込み
# =================================================================

# --- sample_submissionからのラベル順序（列順序を定義）---
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
LABEL2IDX = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
taxonomy = pd.read_csv(TAXONOMY_PATH)
label_to_taxon = dict(zip(taxonomy["primary_label"].astype(str),
                          taxonomy["class_name"].astype(str)))
TAXON_MASKS = {t: np.array([i for i, l in enumerate(PRIMARY_LABELS)
                            if label_to_taxon.get(l, "") == t])
               for t in ["Aves", "Amphibia", "Insecta", "Mammalia", "Reptilia"]}

# --- 焦点録音メタデータ ---
audio_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "audio_cache_meta.csv")
train_df = pd.read_csv(COMP_DIR / "train.csv")
audio_cache_meta = audio_cache_meta.merge(
    train_df[["filename", "secondary_labels"]], on="filename", how="left"
)
audio_cache_meta = audio_cache_meta[
    audio_cache_meta["primary_label"].isin(LABEL2IDX)
].reset_index(drop=True)
print(f"Focal audio cache: {len(audio_cache_meta)} entries")

# --- サウンドスケープウィンドウメタデータ ---
sc_cache_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "soundscape_cache_meta.csv")
sc_cache_meta["label_list"] = sc_cache_meta["label_list"].apply(
    lambda x: x.split(";") if isinstance(x, str) else []
)
print(f"Soundscape cache: {len(sc_cache_meta)} windows")

# --- 正解ラベルからサウンドスケープラベル行列を構築 ---
sc_labels_raw = pd.read_csv(LABELS_PATH).drop_duplicates()
sc_labels_raw["start_sec"] = pd.to_timedelta(sc_labels_raw["start"]).dt.total_seconds().astype(int)

Y_SC = np.zeros((len(sc_cache_meta), NUM_CLASSES), dtype=np.float32)
for i, row in sc_cache_meta.iterrows():
    matches = sc_labels_raw[
        (sc_labels_raw["filename"] == row["filename"]) &
        (sc_labels_raw["start_sec"] == row["start_sec"])
    ]
    for _, m in matches.iterrows():
        for lbl in str(m["primary_label"]).split(";"):
            lbl = lbl.strip()
            if lbl in LABEL2IDX:
                Y_SC[i, LABEL2IDX[lbl]] = 1.0

labeled_sc_mask = Y_SC.sum(axis=1) > 0
print(f"Soundscape labels: {labeled_sc_mask.sum()}/{len(Y_SC)} windows labeled, "
      f"{int(Y_SC.sum())} positives, {int((Y_SC.sum(axis=0) > 0).sum())} species")

# =================================================================
# StratifiedKFold割り当て（希少種の偏り対策）
# =================================================================
audio_for_split = audio_cache_meta.drop_duplicates("original_idx").reset_index(drop=True)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
audio_for_split["fold"] = -1
for fold, (_, val_idx) in enumerate(skf.split(audio_for_split, audio_for_split["primary_label"])):
    audio_for_split.loc[val_idx, "fold"] = fold
audio_cache_meta = audio_cache_meta.merge(
    audio_for_split[["original_idx", "fold"]], on="original_idx", how="left"
)
print(f"\nFocal fold distribution:\n{audio_cache_meta['fold'].value_counts().sort_index()}")

# =================================================================
# GroupKFold割り当て（データリーク対策）
# =================================================================
from sklearn.model_selection import GroupKFold
sc_files = sc_cache_meta[["filename", "site"]].drop_duplicates().reset_index(drop=True)
gkf = GroupKFold(n_splits=N_FOLDS)
sc_files["fold"] = -1
for fold, (_, val_idx) in enumerate(gkf.split(sc_files, groups=sc_files["filename"])):
    sc_files.loc[sc_files.index[val_idx], "fold"] = fold

file_to_fold = dict(zip(sc_files["filename"], sc_files["fold"]))
sc_cache_meta["fold"] = sc_cache_meta["filename"].map(file_to_fold).fillna(-1).astype(int)
print(f"\nSoundscape fold distribution:")
print(sc_cache_meta["fold"].value_counts().sort_index())

# =================================================================
# 希少種のアップサンプリング
# =================================================================
counts = audio_cache_meta["primary_label"].value_counts()
rare_species = counts[counts < MIN_SAMPLE].index
extra_rows = []
for sp in rare_species:
    sp_rows = audio_cache_meta[audio_cache_meta["primary_label"] == sp]
    n_copies = int(np.ceil(MIN_SAMPLE / len(sp_rows))) - 1
    for _ in range(n_copies):
        extra_rows.append(sp_rows)

n_before = len(audio_cache_meta)
if extra_rows:
    audio_cache_meta = pd.concat([audio_cache_meta] + extra_rows, ignore_index=True)
print(f"\nUpsampled {len(rare_species)} rare species (min={MIN_SAMPLE}): "
      f"{n_before} -> {len(audio_cache_meta)} samples")

# =================================================================
# S22（録音した場所の識別子）の除外
# =================================================================
sc_sites = sc_cache_meta["site"].values
non_s22_mask_sc = sc_sites != "S22"
print(f"S22: {(~non_s22_mask_sc).sum()}, non-S22: {non_s22_mask_sc.sum()}")
print("データ読み込み完了")

# --- 変更006 追加 ↓↓↓--------------------------------------------------------------
# # Background noiseファイルリスト初期化
# import soundfile as sf
# _bg_noise_files = []
# if USE_BG_NOISE and BG_NOISE_DIR.is_dir():
#     _bg_noise_files = sorted([
#         str(p) for p in BG_NOISE_DIR.rglob("*")
#         if p.suffix.lower() in (".wav", ".ogg", ".flac", ".mp3")
#     ])
#     print(f"Background noise files: {len(_bg_noise_files)}")
# else:
#     print("Background noise: 無効 or ディレクトリなし")
# --- 変更006 追加 ↑↑↑ -------------------------------------------------------------

# --- 変更007 追加 ↓↓↓--------------------------------------------------------------
# if USE_PSEUDO_007:
#     pseudo_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "pseudo_window_meta.csv")
#     pseudo_meta = pseudo_meta[pseudo_meta["has_expert_label"] == False].reset_index(drop=True)
#     pseudo_meta = pseudo_meta[pseudo_meta["max_logit"] > PSEUDO_LOGIT_MIN].reset_index(drop=True)
#     print(f"Pseudo windows (after threshold): {len(pseudo_meta)}")

#     pseudo_subm = pd.read_csv(PSEUDO_SUBM_PATH)
#     pseudo_subm["filename"] = pseudo_subm["row_id"].str.rsplit("_", n=1).str[0] + ".ogg"
#     pseudo_subm["end_sec"]  = pseudo_subm["row_id"].str.rsplit("_", n=1).str[1].astype(int)
#     pseudo_subm["start_sec"] = pseudo_subm["end_sec"] - 5
#     pseudo_merged = pseudo_meta.merge(
#         pseudo_subm.drop(columns=["row_id", "end_sec"]),
#         on=["filename", "start_sec"], how="left"
#     )
#     score_matrix = pseudo_merged[PRIMARY_LABELS].values.astype(np.float32)
#     # べき乗変換でノイズ抑制
#     Y_PSEUDO_007 = np.power(np.clip(score_matrix, 0, 1), PSEUDO_POWER).astype(np.float32)
#     valid_mask = Y_PSEUDO_007.sum(axis=1) > 0
#     pseudo_meta_007 = pseudo_meta[valid_mask].reset_index(drop=True)
#     Y_PSEUDO_007    = Y_PSEUDO_007[valid_mask]
#     print(f"変更007 Pseudo labels: {len(pseudo_meta_007)} windows")
# --- 変更007 追加 ↑↑↑ -------------------------------------------------------------

# --- 変更008 追加 ↓↓↓--------------------------------------------------------------
# if USE_DISTILL_008:
#     pseudo_meta = pd.read_csv(WAVEFORM_CACHE_DIR / "pseudo_window_meta.csv")
#     pseudo_meta = pseudo_meta[pseudo_meta["has_expert_label"] == False].reset_index(drop=True)
#     pseudo_meta = pseudo_meta[pseudo_meta["max_logit"] > PSEUDO_LOGIT_MIN].reset_index(drop=True)
#     print(f"Pseudo windows (after threshold): {len(pseudo_meta)}")
#
#     pseudo_subm = pd.read_csv(PSEUDO_SUBM_PATH)
#     pseudo_subm["filename"] = pseudo_subm["row_id"].str.rsplit("_", n=1).str[0] + ".ogg"
#     pseudo_subm["end_sec"]  = pseudo_subm["row_id"].str.rsplit("_", n=1).str[1].astype(int)
#     pseudo_subm["start_sec"] = pseudo_subm["end_sec"] - 5
#     pseudo_merged = pseudo_meta.merge(
#         pseudo_subm.drop(columns=["row_id", "end_sec"]),
#         on=["filename", "start_sec"], how="left"
#     )
#     score_matrix = pseudo_merged[PRIMARY_LABELS].values.astype(np.float32)
#     # soft labelそのまま（蒸留）
#     Y_PSEUDO_008 = np.clip(score_matrix, 0, 1).astype(np.float32)
#     valid_mask = Y_PSEUDO_008.sum(axis=1) > 0
#     pseudo_meta_008 = pseudo_meta[valid_mask].reset_index(drop=True)
#     Y_PSEUDO_008    = Y_PSEUDO_008[valid_mask]
#     print(f"変更008 Distill labels: {len(pseudo_meta_008)} windows")
# --- 変更008 追加 ↑↑↑ -------------------------------------------------------------

# --- 変更009 追加 ↓↓↓--------------------------------------------------------------
if USE_PSEUDO_009:
    pseudo_filtered = pd.read_csv(PSEUDO_SUBM_PATH_009)
    # fold_id列でOOF分離（fold kの学習時はfold kのデータを除外）
    # ※学習ループ内でfold_kに応じてフィルタする
    score_matrix_009 = pseudo_filtered[PRIMARY_LABELS].values.astype(np.float32)
    Y_PSEUDO_009 = np.power(np.clip(score_matrix_009, 0, 1), PSEUDO_POWER_009).astype(np.float32)
    pseudo_meta_009 = pseudo_filtered.reset_index(drop=True)
    print(f"変更009 Pseudo labels: {len(pseudo_meta_009)} windows")
# --- 変更009 追加 ↑↑↑ -------------------------------------------------------------


Focal audio cache: 35549 entries
Soundscape cache: 739 windows
Soundscape labels: 739/739 windows labeled, 3122 positives, 75 species

Focal fold distribution:
fold
0    7110
1    7110
2    7110
3    7110
4    7109
Name: count, dtype: int64

Soundscape fold distribution:
fold
0    155
1    137
2    146
3    149
4    152
Name: count, dtype: int64

Upsampled 36 rare species (min=20): 35549 -> 36135 samples
S22: 477, non-S22: 262
データ読み込み完了
変更009 Pseudo labels: 58912 windows


In [ ]:
if DEBUG:
    EPOCHS = 1
    FOLDS = [0]
    audio_cache_meta = audio_cache_meta.groupby("primary_label").head(3).reset_index(drop=True)
    sc_cache_meta = sc_cache_meta.head(50)
    Y_SC = Y_SC[:50]
    non_s22_mask_sc = non_s22_mask_sc[:50]
    print(f"DEBUG MODE: {len(audio_cache_meta)} focal, {len(sc_cache_meta)} sc, "
          f"{EPOCHS} epoch, folds={FOLDS}")

## S3 — モデルアーキテクチャ

1. 以下のクラスを定義する。

- A. MelSpecTransform（波形 → メルスペクトログラム変換）
  - 音声波形を、縦軸=周波数・横軸=時間の2次元画像（メルスペクトログラム）に変換する。
  - 画像認識モデル（EfficientNet）に音声を食わせるための前処理。

- B. SpecAugment（メルスペクトログラムへの変形）
  - 完成したメルスペクトログラムに、周波数・時間方向のマスク（一部を隠す）をかける。
  - オーグメンテーションの一種。モデルが一部の周波数や時間に依存しすぎるのを防ぐ。

- C. PerchTeacher（凍結した教師モデル）
  - 鳥の鳴き声に特化した巨大モデル Perch v2 を、ONNX形式で読み込む。
  - 学習させず（凍結）、5秒の波形を受け取って1536次元の埋め込みを返す教師として使う。
  - ※ONNX … モデルを学習環境に依存しない形式で保存したもの。読み込みが速い。

- D. DistillHead（教師の知識を借りる蒸留ヘッド）
  - バックボーンの特徴をPerch(教師)の1536次元空間に近づける分岐。
  - バックボーン特徴を全体平均（GAP）して線形変換し、教師の埋め込みとMSEで比べることで、少ないデータでも教師が持つ豊かな音の捉え方をバックボーンに学ばせられる。

- E. BirdSEDModel（本体モデル）
  - ABCDを組み合わせた、メルスペクトログラム→234種の確率を出す本体。
  - 内部の処理順：バックボーン → 勾配停止 → GeMプーリング → ボトルネック → SEDアテンション → framewise予測 → clipwise予測（重み付き合計）
  - ※学習時はDistillHeadも並走し、Perch蒸留のMSE損失も計算する。

- ※手法
  - SEDアテンション（一瞬だけ鳴く種を逃さない）
    - フレームごとに234種の「鳴いている度合い」を出し、「どのフレームが重要か」の重みを学習して重み付き合計で5秒全体の予測にまとめる。
    - 短く鳴く種でもその瞬間のフレームに重みが集中するので、長い背景音に薄められずスコアに反映されるため採用。
    - GAPは、短い鳴き声が長い背景音に埋もれて消えてしまうため不採用。
  - GeMFreqPool（周波数方向のプーリング）
    - バックボーンの出力（チャンネル・周波数・時間）から周波数軸だけ潰し、SEDに必要な時間軸を残す。単純平均より特徴的な周波数を強調できるGeMを使用。
  - 蒸留（Perch v2から知識を借りる）
    - 凍結したPerch v2（教師）の1536次元埋め込みに、自分のモデルの特徴をMSEで近づける。
    - 鳥の鳴き声特化・14,795種学習済みの巨大モデルPerchを教師にすることで、少ないデータでも豊かな音の捉え方をバックボーンに移せる。
  - 勾配停止（stop-gradient）
    - `h.detach()`で分類ヘッド側からバックボーンへの勾配を止め、バックボーンはPerchの表現を真似ることに専念、分類ヘッドはその特徴を使うだけ、と役割を分ける。
    - 止めないと「分類で当てたい」と「Perchに似せたい」が引っ張り合い学習が不安定になるため採用。

- ※用語
  - バックボーン
    - 特徴抽出の本体。ここではEfficientNet-B0。画像認識で鍛えた重みを流用する。
  - SED（Sound Event Detection・音響イベント検出）アテンション
    - フレームごとに234種それぞれの「鳴いている度合い」を出し、重要なフレームに重みを集中させて5秒全体の予測にまとめる仕組み。弱ラベルから鳴いた瞬間を自動で拾うために使う。
  - GAP（Global Average Pooling・全体平均プーリング）
    - 特徴マップ全体を1つの数値に平均すること。シンプルだが鳴いた瞬間が薄まる欠点がある。
  - GeM（Generalized Mean Pooling・一般化平均プーリング）
    - 平均と最大値の中間で鋭さを調整できるプーリング。pを学習で決める。
  - MSE（Mean Squared Error・平均二乗誤差）
    - 2つの値の差を二乗して平均した数値。差が大きいほど大きくなる。
    - 蒸留では「自分のモデルの埋め込み」と「Perchの埋め込み」の差をMSEで測り、小さくなるよう学習する。
  - 蒸留
    - 大きな教師モデルの知識を、小さな生徒モデルに移す手法。
      直接ラベルで学ぶより、教師の内部表現に近づけることで少ないデータでも効く。
  - ONNX
    - モデルを学習フレームワークに依存しない形式で保存する規格。
      PyTorchで学習してONNXで保存し、高速に推論できる。
  - framewise / clipwise
    - framewise：5秒を細かく切ったフレームごとの予測。「何秒目に鳴いたか」が分かる。
    - clipwise：5秒クリップ全体を1つにまとめた予測。これが最終出力として提出に使われる。

In [ ]:
# =================================================================
# S3 -- 評価ユーティリティ + メル変換 + SEDモデル
# =================================================================

def compute_macro_auc(y_true, y_pred, mask=None, class_mask=None):
    """評価可能な種全体のマクロ平均AUC。"""
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    if class_mask is not None:
        y_true, y_pred = y_true[:, class_mask], y_pred[:, class_mask]
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        try:
            aucs.append(roc_auc_score(col, y_pred[:, c]))
        except ValueError:
            continue
    return (np.mean(aucs) if aucs else float("nan")), len(aucs)

def full_eval(y_true, y_pred, ns22, tm):
    r = {}
    a, n = compute_macro_auc(y_true, y_pred)
    r["macro_auc_all"], r["n_all"] = round(a, 4), n
    a, n = compute_macro_auc(y_true, y_pred, mask=ns22)
    r["non_s22_macro"], r["n_ns22"] = round(a, 4), n
    for t, cm in tm.items():
        a, n = compute_macro_auc(y_true, y_pred, mask=ns22, class_mask=cm)
        r[f"non_s22_{t}"] = round(a, 4)
    return r

# ------------------------------------------------------------------
# GPU メルスペクトログラム
# ------------------------------------------------------------------
class MelSpecTransform(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
            n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
        )
        self.db_transform = torchaudio.transforms.AmplitudeToDB(top_db=80)

    def forward(self, waveform):
        return self.db_transform(self.mel_spec(waveform))

# ------------------------------------------------------------------
# GPU SpecAugment
# ------------------------------------------------------------------
class SpecAugment(nn.Module):
    def __init__(self):
        super().__init__()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=FREQ_MASK_PARAM)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=TIME_MASK_PARAM)

    def forward(self, mel):
        for _ in range(NUM_FREQ_MASKS):
            mel = self.freq_mask(mel)
        for _ in range(NUM_TIME_MASKS):
            mel = self.time_mask(mel)
        return mel

# ------------------------------------------------------------------
# 凍結Perch教師モデル -- ONNX推論、勾配なし
# ------------------------------------------------------------------
import onnxruntime as ort

class PerchTeacher:
    """ONNX経由の凍結Perch v2。5秒波形を受け取り1536次元埋め込みを返す。
    教師モデルは更新されない -- 安定した蒸留ターゲットを提供する。"""

    def __init__(self, onnx_path, device_str="cuda"):
        providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] \
            if device_str == "cuda" else ["CPUExecutionProvider"]
        self.session = ort.InferenceSession(str(onnx_path), providers=providers)
        self.input_name = self.session.get_inputs()[0].name
        self._out_names = [o.name for o in self.session.get_outputs()]
        self._embed_idx = None
        for i, o in enumerate(self.session.get_outputs()):
            if o.shape and o.shape[-1] == PERCH_EMBED_DIM:
                self._embed_idx = i
                break
        if self._embed_idx is None:
            self._embed_idx = 1
        print(f"Perch ONNX loaded: embed_idx={self._embed_idx}")

    @torch.no_grad()
    def embed(self, waveforms_5s):
        """waveforms_5s: (B, 160000) float32、(B, 1536)埋め込みを返す。"""
        wav_np = waveforms_5s.cpu().numpy()
        results = self.session.run(None, {self.input_name: wav_np})
        return torch.from_numpy(results[self._embed_idx]).float()

# ------------------------------------------------------------------
# 蒸留ヘッド: GAP + Perch埋め込み空間への線形変換
# ------------------------------------------------------------------
class DistillHead(nn.Module):
    """GAP + 線形変換でバックボーン特徴量をPerchの1536次元空間に射影する。"""
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)

    def forward(self, feature_map):
        gap = feature_map.mean(dim=[2, 3])   # (B, C, F, T) -> (B, C)
        return self.proj(gap)                 # (B, embed_dim)

# ------------------------------------------------------------------
# SEDモデル V2: GeMFreq + ボトルネック + AttBlock（推奨）
# ------------------------------------------------------------------
class GeMFreqPool(nn.Module):
    """Generalized Mean pooling over frequency. Learnable p starts at 3.0
    (sharper than mean, softer than max). Lets the model emphasize
    frequency bands where species vocalize."""
    def __init__(self, p_init=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(float(p_init)))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        # --- 変更001 削除 ↓↓↓--------------------------------------------------------------
        x = x.mean(dim=2)
        # --- 変更001 削除 ↑↑↑ -------------------------------------------------------------
        # --- 変更001 追加 ↓↓↓--------------------------------------------------------------
        # EfficientNet-B0からEfficientNet-V2-Sに変更する。
        # パラメータ数が約4倍になり、特徴抽出能力が向上する。
        # 一方でGPUメモリ消費が増えるため、BATCHサイズの調整が必要な場合がある。
        # x = x.mean(dim=3)
        # --- 変更001 追加 ↑↑↑ -------------------------------------------------------------

        return x.pow(1.0 / p)

class BirdSEDModel(nn.Module):
    """SED model with 1st-place-inspired design: https://www.kaggle.com/code/nikitababich/birdclef2025-1st-place-inference
    - GeMFreq pooling (learnable, sharper than mean)
    - 512-dim bottleneck with dropout
    - Attention-weighted clip logits from frame logits
    - Distillation: GAP+Linear branch for MSE to Perch
    - Stop gradient: backbone trains from distillation only
    """
    def __init__(self, backbone_name=BACKBONE_NAME, num_classes=NUM_CLASSES,
                 drop_path_rate=0.15,
                 hidden_dim=512):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=True, in_chans=1,
            num_classes=0, global_pool="", drop_path_rate=drop_path_rate,
        )
        with torch.no_grad():
            n_tf = TRAIN_SAMPLES // HOP_LENGTH + 1
            dummy = torch.randn(1, 1, N_MELS, n_tf)
            feat = self.backbone(dummy)
            self.backbone_dim = feat.shape[1]
            print(f"V2 backbone: {tuple(feat.shape)}  (C={self.backbone_dim})")

        self.gem_freq = GeMFreqPool(p_init=3.0)
        self.dense = nn.Sequential(
            nn.Dropout(0.25),
            nn.Linear(self.backbone_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.att = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(hidden_dim, num_classes, kernel_size=1, bias=True)
        nn.init.xavier_uniform_(self.att.weight)
        nn.init.xavier_uniform_(self.cla.weight)
        self.att.bias.data.fill_(0.)
        self.cla.bias.data.fill_(0.)
        if USE_PERCH_DISTILL:
            self.distill_head = DistillHead(self.backbone_dim, PERCH_EMBED_DIM)

    def forward(self, x, return_framewise=False, return_distill=False):
        h = self.backbone(x)
        distill_emb = None
        if return_distill and hasattr(self, 'distill_head'):
            distill_emb = self.distill_head(h)

        # 勾配停止: SEDヘッドはバックボーンを更新しない
        h_cls = h.detach() if USE_PERCH_DISTILL else h

        h_cls = self.gem_freq(h_cls)            # (B, C, T)
        h_cls = h_cls.permute(0, 2, 1)          # (B, T, C)
        h_cls = self.dense(h_cls)               # (B, T, 512)
        h_cls = h_cls.permute(0, 2, 1)          # (B, 512, T)

        norm_att = torch.softmax(torch.tanh(self.att(h_cls)), dim=-1)
        framewise_logits = self.cla(h_cls)
        clip_logits = torch.sum(norm_att * framewise_logits, dim=2)

        fw = framewise_logits.permute(0, 2, 1) if return_framewise else None
        if return_framewise and return_distill: return clip_logits, fw, distill_emb
        elif return_framewise: return clip_logits, fw
        elif return_distill: return clip_logits, distill_emb
        return clip_logits

def make_model():
        return BirdSEDModel(BACKBONE_NAME).to(device)

print("モデル定義準備完了")

モデル定義準備完了


## S4 — データパイプライン

1. 以下の処理を定義する。

- A. 波形の読み込み（load_int16 / load_focal / load_sc_waveform_from）
  - 音声を事前にint16形式の`.pt`ファイルにキャッシュして保存してある。
  - 学習ループで呼ばれるたびに`.pt`を読み込み、float32（-1〜1）に変換して使う。
  - ※int16はfloat32の半分のサイズ。キャッシュしておくことで毎回音声を変換する処理を省き、学習を高速化する。

- B. オーグメンテーション（apply_aug）
  - 波形に確率0.5でランダムな変形をかける。以下の3種類。
    - ゲインジッタ（±6dB）… 音量をランダムに上下し、マイクとの距離の違いを再現。
    - 加算ノイズ（10〜30dB SNR）… 雑音を足し、実環境の騒がしさを再現。
    - タイムシフト（±0.5秒）… 鳴くタイミングを少しずらす。

- C. MixUpプール構築（sc_mixup_sources）
  - 焦点-サウンドスケープMixUpに使うラベル付きサウンドスケープウィンドウを事前にリスト化。
  - 学習中に焦点音声とランダムに組み合わせて使う。

- D. FocalDS（焦点音声のデータセット）
  - 焦点音声を学習ループに流すためのデータセット。
  - 以下の2種類のMixUpを内部で実行する。
    - 焦点-焦点MixUp … 2つの焦点音声を重ねて中間の音を作る。ラベルは和集合。
    - 焦点-サウンドスケープMixUp … 焦点とサウンドスケープを重ねて中間の音を作る。ドメインギャップ対策。

- ※手法
  - 波形キャッシュ（int16）
    - 毎回音声ファイルを開いて変換すると重い。事前にint16の`.pt`に変換して保存しておくことで、学習ループ内の読み込みを高速化する。
  - オーグメンテーション（3種）
    - 同じ音声でも毎回少しずつ違う変形がかかるので、丸暗記を防いで汎化を上げる。希少種アップサンプリングと組み合わせると、複製した音声も「少しずつ違うバージョン」になる（S2で話した内容）。
  - 焦点-焦点MixUp
    - 2つの焦点音声を重ねて人工的に「複数種が同時に鳴いている音」を作る。本番のサウンドスケープに近い複雑な音への耐性をつける。
  - 焦点-サウンドスケープMixUp
    - 焦点（きれいな音）とサウンドスケープ（ごちゃごちゃの音）を重ねて中間の音を作る。S2で話したドメインギャップ対策②の実体がここ。

- ※用語
  - int16 / float32
    - int16は-32768〜32767の整数で音を表す形式。float32は小数で表す形式。
    - int16はfloat32の半分のサイズなので、キャッシュの保存容量と読み込み速度が改善する。
  - SNR（Signal-to-Noise Ratio・信号対雑音比）
    - 信号（鳴き声）と雑音の比率。dBで表す。値が小さいほど雑音が大きい（うるさい環境）。
  - MixUp
    - 2つの音声を重み付きで重ねて新しい音声を作るオーグメンテーション手法。ラベルも同じ重みで混ぜる（ソフト）か、和集合にする（ハード）かを選べる。

In [ ]:
# =================================================================
# S4 -- データパイプライン
# =================================================================

def load_int16(path):
    """int16波形テンソルをfloat32の[-1, 1]に読み込む。"""
    waveform_int16 = torch.load(path, map_location="cpu")
    return waveform_int16.float() / 32767.0

_FC = {}
def load_focal(p):
    """シンプルなLRUキャッシュで焦点波形を読み込む。"""
    if p in _FC: return _FC[p]
    pp = WAVEFORM_CACHE_LOCAL_DIR / p
    if not pp.exists(): return None
    a = load_int16(pp).numpy()
    if len(_FC) >= 2000:
        _FC.pop(next(iter(_FC)))
    _FC[p] = a
    return a

_SC_CACHE = {}
def load_sc_waveform_from(cache_dir, cache_file):
    """LRUキャッシュでサウンドスケープ波形を読み込む。"""
    key = str(cache_dir / cache_file)
    if key in _SC_CACHE: return _SC_CACHE[key]
    pp = cache_dir / cache_file
    if not pp.exists(): return None
    a = load_int16(pp).numpy()
    if len(_SC_CACHE) >= 200:
        _SC_CACHE.pop(next(iter(_SC_CACHE)))
    _SC_CACHE[key] = a
    return a

def extract_chunk_np(waveform, start_sample, n_samples):
    """チャンクを抽出する。録音が短すぎる場合は左パディングする。"""
    total = len(waveform)
    if total <= n_samples:
        return np.pad(waveform, (n_samples - total, 0))
    end = start_sample + n_samples
    if end > total:
        start_sample = max(0, total - n_samples)
    return waveform[start_sample:start_sample + n_samples]

def apply_aug(w):
    """シンプルな波形オーグメンテーション: ゲインジッタ + ノイズ + シフト。"""
    # --- 変更006 削除 ↓↓↓--------------------------------------------------------------
    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))
    return w
    # --- 変更006 削除 ↑↑↑ -------------------------------------------------------------

    if np.random.random() < AUG_PROB:
        w = w * (10 ** (np.random.uniform(*AUG_GAIN_DB_RANGE) / 20))
    if np.random.random() < AUG_PROB:
        sp = (w ** 2).mean()
        if sp > 1e-10:
            w = w + np.random.randn(*w.shape).astype(w.dtype) * np.sqrt(
                sp / (10 ** (np.random.uniform(*AUG_NOISE_SNR_DB_RANGE) / 10)))

    # --- 変更006 追加 ↓↓↓--------------------------------------------------------------
    # if USE_BG_NOISE and np.random.random() < BG_NOISE_PROB and _bg_noise_files:
    #     bg_path = _bg_noise_files[np.random.randint(len(_bg_noise_files))]
    #     try:
    #         bg, bg_sr = sf.read(bg_path, dtype='float32')
    #         if bg.ndim > 1: bg = bg.mean(axis=1)
    #         if bg_sr != SR:
    #             import scipy.signal
    #             bg = scipy.signal.resample(bg, int(len(bg) * SR / bg_sr))
    #         n = len(w)
    #         if len(bg) < n:
    #             bg = np.tile(bg, int(np.ceil(n / len(bg))))
    #         start = np.random.randint(0, len(bg) - n + 1)
    #         bg = bg[start:start + n].astype(np.float32)
    #         sp = (w ** 2).mean()
    #         bg_sp = (bg ** 2).mean()
    #         if sp > 1e-10 and bg_sp > 1e-10:
    #             snr = 10 ** (np.random.uniform(*BG_NOISE_SNR_DB_RANGE_BG) / 10)
    #             bg = bg * np.sqrt(sp / (snr * bg_sp))
    #             w = w + bg
    #     except Exception:
    #         pass
    # return w
    # --- 変更006 追加 ↑↑↑ -------------------------------------------------------------

# ------------------------------------------------------------------
# サウンドスケープMixUpプールの構築（ラベル付きウィンドウのみ）
# ------------------------------------------------------------------
sc_mixup_sources = []
_sc_file_meta = pd.read_csv(WAVEFORM_CACHE_LOCAL_DIR  / "soundscape_file_meta.csv")
_sc_file_dict = dict(zip(_sc_file_meta["filename"], _sc_file_meta["cache_file"]))
_labeled_rows = []
for i in range(len(sc_cache_meta)):
    row = sc_cache_meta.iloc[i]
    if Y_SC[i].sum() > 0:
        cf = _sc_file_dict.get(row["filename"])
        if cf is not None:
            _labeled_rows.append({
                "filename": row["filename"], "start_sec": int(row["start_sec"]),
                "cache_file": cf, "label_idx": i, "fold": int(row.get("fold", -1)),
            })
if _labeled_rows:
    _labeled_meta = pd.DataFrame(_labeled_rows)
    sc_mixup_sources.append((WAVEFORM_CACHE_DIR, _labeled_meta, Y_SC))
    print(f"SC MixUp pool: {len(_labeled_meta)} labeled windows")

# ------------------------------------------------------------------
# FocalDS -- 焦点-サウンドスケープMixUp（※Markdown S2を参照）
# ------------------------------------------------------------------
class FocalDS(Dataset):
    """焦点録音データセット。(波形, ラベル, 重み, マスク, ソースタグ)を返す。"""
    def __init__(self, df, l2i, secondary_lookup=None,
                 sc_mixup_sources=None, fold_k=None, aug=False):
        self.df, self.l2i, self.aug = df.reset_index(drop=True), l2i, aug
        self.secondary_lookup = secondary_lookup
        self.sc_mixup_sources = sc_mixup_sources
        self.fold_k = fold_k

    def __len__(self): return len(self.df)

    def _load_chunk(self, r):
        w = load_focal(r["cache_file"])
        if w is None: return None, None
        if self.aug:
            start = np.random.randint(0, max(1, len(w) - TRAIN_SAMPLES + 1)) if len(w) > TRAIN_SAMPLES else 0
        else:
            start = int(r.get("start_sec", 0)) * SR
        ch = extract_chunk_np(w, start, TRAIN_SAMPLES)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        if str(r["primary_label"]) in self.l2i:
            lb[self.l2i[str(r["primary_label"])]] = 1.0
        if self.secondary_lookup is not None and "original_idx" in self.df.columns:
            for s in self.secondary_lookup.get(int(r["original_idx"]), []):
                if s in self.l2i: lb[self.l2i[s]] = 1.0
        return ch, lb

    def __getitem__(self, i):
        r1 = self.df.iloc[i]
        ch1, lb1 = self._load_chunk(r1)
        if ch1 is None:
            return (torch.zeros(1, TRAIN_SAMPLES), torch.zeros(NUM_CLASSES),
                    torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal_missing")

        # 焦点-焦点MixUp
        if USE_FOCAL_MIXUP and self.aug and np.random.random() < MIXUP_PROB:
            ch2 = None
            for _ in range(3):
                j = np.random.randint(len(self.df))
                ch2, lb2 = self._load_chunk(self.df.iloc[j])
                if ch2 is not None: break
            if ch2 is not None:
                lam = 0.5
                ch_mix = (lam * ch1 + (1 - lam) * ch2).astype(np.float32)
                if self.aug: ch_mix = apply_aug(ch_mix)
                lb = np.maximum(lb1, lb2) if MIXUP_HARD else (lam * lb1 + (1 - lam) * lb2)
                return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                        torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")

        # 焦点-サウンドスケープMixUp
        if (USE_FOCAL_SC_MIXUP and self.aug and self.sc_mixup_sources
                and np.random.random() < FOCAL_SC_MIXUP_PROB):
            src_idx = np.random.randint(len(self.sc_mixup_sources))
            cache_dir, meta_df_sc, labels = self.sc_mixup_sources[src_idx]
            eligible = meta_df_sc[meta_df_sc["fold"] != self.fold_k] if self.fold_k is not None else meta_df_sc
            if len(eligible) > 0:
                sc_row = eligible.iloc[np.random.randint(len(eligible))]
                sc_wav = load_sc_waveform_from(cache_dir, sc_row["cache_file"])
                if sc_wav is not None and len(sc_wav) >= TRAIN_SAMPLES:
                    sc_chunk = extract_chunk_np(sc_wav, int(sc_row["start_sec"]) * SR, TRAIN_SAMPLES)
                    lam = 0.5
                    ch_mix = (lam * ch1 + (1 - lam) * sc_chunk).astype(np.float32)
                    if self.aug: ch_mix = apply_aug(ch_mix)
                    lb_sc = labels[int(sc_row["label_idx"])].astype(np.float32)
                    lb = np.maximum(lb1, lb_sc) if MIXUP_HARD else lam * lb1 + (1 - lam) * lb_sc
                    return (torch.from_numpy(ch_mix).unsqueeze(0), torch.from_numpy(lb),
                            torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")

        # MixUpなし
        if self.aug: ch1 = apply_aug(ch1)
        return (torch.from_numpy(ch1.astype(np.float32)).unsqueeze(0),
                torch.from_numpy(lb1),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "focal")

# ------------------------------------------------------------------
# ScDS -- ラベル付きサウンドスケープウィンドウ
# ------------------------------------------------------------------
class ScDS(Dataset):
    def __init__(self, Y, sc_df, aug=False):
        self.Y, self.df, self.aug = Y, sc_df.reset_index(drop=True), aug
    def __len__(self): return len(self.Y)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        wav_full = load_sc_waveform_from(WAVEFORM_CACHE_DIR, row.get("cache_file")) \
                   if row.get("cache_file") else None
        if wav_full is None:
            wav_t = torch.zeros(1, TRAIN_SAMPLES)
        else:
            chunk = extract_chunk_np(wav_full, int(row["start_sec"]) * SR, TRAIN_SAMPLES)
            if self.aug: chunk = apply_aug(chunk)
            wav_t = torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0)
        return (wav_t, torch.from_numpy(self.Y[i].astype(np.float32)),
                torch.ones(NUM_CLASSES), torch.ones(NUM_CLASSES), "sc")

# ------------------------------------------------------------------
# 焦点二次ラベルの読み込み
# ------------------------------------------------------------------
focal_secondary_labels = None
if USE_FOCAL_SECONDARY:
    focal_secondary_labels = {}
    for idx, row in train_df.iterrows():
        sec = row.get("secondary_labels", "")
        if pd.isna(sec) or sec in ("", "[]"): continue
        try:
            sec_list = eval(sec) if isinstance(sec, str) else []
        except: continue
        valid = [s for s in sec_list if s in LABEL2IDX]
        if valid: focal_secondary_labels[idx] = valid
    print(f"Focal secondary labels: {len(focal_secondary_labels)} files")

# ------------------------------------------------------------------
# マルチソースバッチサンプラー
# ------------------------------------------------------------------
class MixSamp(torch.utils.data.Sampler):
    """ソースごとのシェアでバッチ構成を制御する。"""
    def __init__(self, sizes, names, shares, bs, nst, seed=0, pseudo_weights=None, pseudo_idx=None):
        self.sizes, self.names, self.bs, self.nst = sizes, names, bs, nst
        self.rng = np.random.default_rng(seed)
        per_src = [max(1, int(round(bs * shares.get(n, 0.0)))) for n in names]
        total = sum(per_src)
        if total != bs:
            per_src[int(np.argmax(per_src))] += (bs - total)
        self.per_src = per_src
        self.offsets = [0]
        for s in sizes[:-1]:
            self.offsets.append(self.offsets[-1] + s)
        # --- 変更009: WeightedPseudoSampler ---
        self.pseudo_weights = pseudo_weights
        self.pseudo_idx = pseudo_idx
        if pseudo_weights is not None:
            w = pseudo_weights.astype(np.float64)
            self.pseudo_probs = w / w.sum()
    def __len__(self): return self.nst
    def __iter__(self):
        for _ in range(self.nst):
            batch = []
            for idx, (off, size, n) in enumerate(zip(self.offsets, self.sizes, self.per_src)):
                if n <= 0 or size <= 0: continue
                # --- 変更009: pseudoはWeightedサンプリング ---
                if self.pseudo_weights is not None and idx == self.pseudo_idx:
                    idxs = self.rng.choice(size, size=n, replace=True, p=self.pseudo_probs)
                else:
                    idxs = self.rng.integers(0, size, size=n)
                batch.extend([off + int(i) for i in idxs])
            self.rng.shuffle(batch)
            yield batch

def collate_m(batch):
    return (torch.stack([b[0] for b in batch]),
            torch.stack([b[1] for b in batch]),
            torch.stack([b[2] for b in batch]),
            torch.stack([b[3] for b in batch]),
            [b[4] for b in batch])

def mk_sw(sr):
    """サンプルごとのソース重みテンソル。"""
    return torch.tensor([SOURCE_WEIGHTS.get(s, 0.0) for s in sr], dtype=torch.float32)

print("データパイプライン準備完了")

SC MixUp pool: 739 labeled windows
Focal secondary labels: 4372 files
データパイプライン準備完了


## S5 — 学習ループ

1. 各エポックで以下を繰り返す。

- A. バッチのサンプリング（MixSamp）
  - 焦点80% / サウンドスケープ10% / 疑似ラベル10%の構成でバッチを作る（変更009）。
  - ※S2で話したドメインギャップ対策①の実体がここ。

- B. 前処理（メルスペクトログラム変換 + SpecAugment）
  - バッチの波形をGPU上でメルスペクトログラムに変換し、SpecAugmentをかける。
  - GPU上でやることで、CPUで事前変換するより柔軟に変形を変えられる。

- C. 順伝播
  - メルスペクトログラムをモデルに通し、clip_logits・framewise・distill_embを得る。
  - バックボーン → 勾配停止 → GeMプーリング → SEDアテンション → clipwise予測、という流れ（S3参照）。

- D. 損失計算
  - 分類損失（BCE）… clipwiseとframewiseの両方で正解ラベルと比べる。0.5:0.5でブレンド。
  - 蒸留損失（MSE）… distill_embをPerchの埋め込みに近づける。
  - 合計損失 = 分類損失 + α × 蒸留損失（α=ALPHA_DISTILL）。

- E. 逆伝播・重み更新
  - 損失をバックボーンに向かって伝播させ、各パラメータの勾配を計算して重みを更新する。
  - 勾配クリッピング（ノルム1.0）… 勾配が大きくなりすぎて学習が発散するのを防ぐ。

- F. 学習率スケジュール
  - 最初の2エポックは線形ウォームアップ（低い学習率から徐々に上げる）。
  - その後コサイン減衰（1e-6まで徐々に下げる）。

- G. 検証
  - ホールドアウトしたサウンドスケープウィンドウで、clip・fmax・blendの3種の予測を評価。
  - S22は除外（S2で話した内容）。

- ※手法
  - BCE（Binary Cross Entropy・二値交差エントロピー）
    - 多ラベル分類の損失。234種それぞれについて「鳴いているか否か」を独立に当てる。
    - clipwiseとframewiseの両方で計算して0.5:0.5でブレンド。framewiseを加えることで「どの瞬間に鳴いたか」も学習に活かせる。
  - 学習率スケジュール（ウォームアップ→コサイン減衰）
    - 最初から大きい学習率で動かすと不安定になりやすいため、ウォームアップで徐々に上げる。その後コサイン減衰で少しずつ下げ、終盤は細かく調整する。
  - 勾配クリッピング
    - 損失が急に大きくなったとき、勾配も大きくなって重みが一気に動きすぎる（学習の発散）。ノルム1.0を上限にクリップすることで安定させる。
  - blend予測（clip + fmax）
    - clipwise（アテンション重み付き合計）とfmax（framewiseの最大値）を0.5:0.5で平均した予測。clipだけより一瞬の鳴き声を強調しやすい。

- ※用語
  - エポック
    - 全学習データを1周すること。複数エポック繰り返して徐々に学習を深める。
  - バッチ
    - 一度に処理するサンプルのまとまり。バッチ単位で損失を計算し重みを更新する。
  - 逆伝播（バックプロパゲーション）
    - 損失を出力側から入力側（バックボーン）に向かって伝播させ、各パラメータの勾配を計算する処理。
  - 学習率
    - 1回の重み更新でどれくらい動かすかの大きさ。大きすぎると発散、小さすぎると収束が遅い。
  - α（ALPHA_DISTILL）
    - 蒸留損失の重み。分類損失と蒸留損失のバランスを調整するパラメータ。

In [ ]:
# =================================================================
# S5 -- 学習
# 1エポック = 全学習データを1周すること
# バッチ = 一度に処理するサンプルのまとまり
# =================================================================

def _load_val_waveforms(val_sc_df):
    """検証波形を読み込む（常に5秒）。"""
    sc_file_meta = pd.read_csv(WAVEFORM_CACHE_LOCAL_DIR  / "soundscape_file_meta.csv")
    sc_file_dict = dict(zip(sc_file_meta["filename"], sc_file_meta["cache_file"]))
    wavs = []
    for _, row in val_sc_df.iterrows():
        cf = sc_file_dict.get(row["filename"])
        if cf is not None:
            w = load_sc_waveform_from(WAVEFORM_CACHE_LOCAL_DIR, cf)
            if w is not None:
                chunk = extract_chunk_np(w, int(row["start_sec"]) * SR, VAL_SAMPLES)
                wavs.append(torch.from_numpy(chunk.astype(np.float32)).unsqueeze(0))
            else: wavs.append(torch.zeros(1, VAL_SAMPLES))
        else: wavs.append(torch.zeros(1, VAL_SAMPLES))
    return wavs

def _predict_from_waveforms(model, mel_transform, wav_list, batch_size=64):
    """推論: メル -> モデル -> シグモイド。蒸留ヘッドは使用しない。"""
    model.eval()
    preds_clip, preds_fmax, preds_blend = [], [], []
    with torch.no_grad():
        for s in range(0, len(wav_list), batch_size):
            batch = torch.stack(wav_list[s:s+batch_size]).to(device)
            mel = mel_transform(batch)
            B = mel.size(0)
            for i in range(B):
                mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
            with autocast():  # 混合精度演算: FP16を使って高速化・メモリ削減
                clip_logits, framewise = model(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
                p_clip = torch.sigmoid(clip_logits).cpu().numpy()
                p_fmax = torch.sigmoid(frame_max).cpu().numpy()
                p_blend = 0.5 * p_clip + 0.5 * p_fmax
            preds_clip.append(p_clip); preds_fmax.append(p_fmax); preds_blend.append(p_blend)
    return {"clip": np.concatenate(preds_clip), "fmax": np.concatenate(preds_fmax),
            "blend": np.concatenate(preds_blend)}

def build_active_datasets(fold_k):
    items = []
    if USE_FOCAL:
        fds = FocalDS(audio_cache_meta[audio_cache_meta["fold"] != fold_k],
                      LABEL2IDX, secondary_lookup=focal_secondary_labels,
                      sc_mixup_sources=sc_mixup_sources if USE_FOCAL_SC_MIXUP else None,
                      fold_k=fold_k, aug=True)
        items.append(("focal", fds, len(fds)))
    if USE_LABELED_SC:
        vm = sc_cache_meta["fold"].values == fold_k
        sc_train_df = sc_cache_meta[~vm].reset_index(drop=True)
        Y_tr = Y_SC[~vm]
        sds = ScDS(Y_tr, sc_train_df, aug=True)
        items.append(("sc", sds, len(sds)))

    # --- 変更007 追加 ↓↓↓--------------------------------------------------------------
    # if USE_PSEUDO_007 and Y_PSEUDO_007 is not None:
    #     pds = ScDS(Y_PSEUDO_007, pseudo_meta_007, aug=True)
    #     items.append(("pseudo", pds, len(pds)))
    # --- 変更007 追加 ↑↑↑ -------------------------------------------------------------

    # --- 変更008 追加 ↓↓↓--------------------------------------------------------------
    # if USE_DISTILL_008 and Y_PSEUDO_008 is not None:
    #     pds = ScDS(Y_PSEUDO_008, pseudo_meta_008, aug=True)
    #     items.append(("pseudo", pds, len(pds)))
    # --- 変更008 追加 ↑↑↑ -------------------------------------------------------------

    # --- 変更009 追加 ↓↓↓--------------------------------------------------------------
    if USE_PSEUDO_009 and pseudo_meta_009 is not None:
        mask = pseudo_meta_009["fold_id"] != fold_k
        meta_k = pseudo_meta_009[mask].reset_index(drop=True)
        Y_k = Y_PSEUDO_009[mask.values]
        if len(meta_k) > 0:
            pds = ScDS(Y_k, meta_k, aug=True)
            items.append(("pseudo", pds, len(pds)))
    # --- 変更009 追加 ↑↑↑ -------------------------------------------------------------

    return items

def train_fold(fold_k):
    vm = sc_cache_meta["fold"].values == fold_k
    Y_val = Y_SC[vm]
    ns22_val = non_s22_mask_sc[vm]
    val_sc_df = sc_cache_meta[vm].reset_index(drop=True)

    active = build_active_datasets(fold_k)
    names, datasets, sizes = zip(*active)
    mds = ConcatDataset(list(datasets))
    nst = max(100, int(sum(sizes) / BATCH))

    print(f"  Streams: {dict(zip(names, sizes))}  steps/ep: {nst}")

    m = make_model()
    mel_transform = MelSpecTransform().to(device)
    spec_augment = SpecAugment().to(device)
    perch_teacher = PerchTeacher(PERCH_ONNX_PATH,
                                  "cuda" if torch.cuda.is_available() else "cpu") \
                    if USE_PERCH_DISTILL else None

    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WD)
    scaler = GradScaler()
    warmup_steps = nst * WARMUP_EPOCHS
    total_steps  = nst * EPOCHS
    warmup_sched = torch.optim.lr_scheduler.LinearLR(opt, start_factor=1/25, end_factor=1.0,
                                                      total_iters=warmup_steps)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps - warmup_steps,
                                                               eta_min=1e-6)
    sch = torch.optim.lr_scheduler.SequentialLR(opt, schedulers=[warmup_sched, cosine_sched],
                                                 milestones=[warmup_steps])

    history = {"ep": [], "train_loss": [], "cls_loss": [], "dist_loss": [],
               "macro": [], "ns22_macro": [],
               "ns22_Aves": [], "ns22_Amphibia": [], "ns22_Insecta": [], "ns22_Mammalia": [],
               "val_preds": []}
    best_ns22, best_state_ns22 = -1.0, None
    best_macro, best_state_macro = -1.0, None

    # --- 変更004 追加 ↓↓↓--------------------------------------------------------------
    # Checkpoint soup用のリスト（後半1/3 epochの重みを蓄積）
    soup_states = []
    # --- 変更004 追加 ↑↑↑ -------------------------------------------------------------

    val_wavs = _load_val_waveforms(val_sc_df)

    for ep in range(EPOCHS):
        print(f"\n{'-'*50}")
        print(f"{ts()} EPOCH {ep}")
        m.train()
        # --- 変更009 追加 ↓↓↓--------------------------------------------------------------
        # pseudoデータにWeightedPseudoSamplerを適用
        if USE_PSEUDO_009 and "pseudo" in names:
            pseudo_idx = list(names).index("pseudo")
            mask_k = pseudo_meta_009["fold_id"] != fold_k
            pseudo_weights = Y_PSEUDO_009[mask_k.values].max(axis=1)
            smp = MixSamp(list(sizes), list(names), SHARES, BATCH, nst, seed=42 + ep,
                          pseudo_weights=pseudo_weights, pseudo_idx=pseudo_idx)
        else:
            # ドメインギャップ対策（※Markdown S2を参照）
            smp = MixSamp(list(sizes), list(names), SHARES, BATCH, nst, seed=42 + ep)
        # --- 変更009 追加 ↑↑↑ -------------------------------------------------------------
        tl = DataLoader(mds, batch_sampler=smp, collate_fn=collate_m,
                        num_workers=0, pin_memory=True)
        el, el_cls, el_dist, nb_count = 0.0, 0.0, 0.0, 0
        t0 = time.time()

        for wav, lb, wt, mk, sr in tl:
            wav, lb, wt, mk = wav.to(device), lb.to(device), wt.to(device), mk.to(device)
            sw = mk_sw(sr).to(device)

            with torch.no_grad():
                mel = mel_transform(wav)
                B = mel.size(0)
                for i in range(B):
                    mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
                mel = spec_augment(mel)

            with autocast():  # 混合精度演算: FP16を使って高速化・メモリ削減
                if USE_PERCH_DISTILL:
                    clip_logits, framewise, distill_emb = m(mel, return_framewise=True,
                                                            return_distill=True)
                else:
                    clip_logits, framewise = m(mel, return_framewise=True)

                frame_max_logits = framewise.max(dim=1).values

                # 分類損失
                bce_clip = F.binary_cross_entropy_with_logits(clip_logits, lb, reduction="none")
                bce_frame = F.binary_cross_entropy_with_logits(frame_max_logits, lb, reduction="none")
                bce = 0.5 * bce_clip + 0.5 * bce_frame
                ps = (bce * wt * mk).sum(1) / (mk.sum(1) + 1e-8)
                cls_loss = (ps * sw).mean()

                # 蒸留損失
                if USE_PERCH_DISTILL and perch_teacher is not None:
                    with torch.no_grad():
                        wav_5s = wav.squeeze(1)
                        N = wav_5s.shape[1]
                        if N > 160000:
                            start = (N - 160000) // 2
                            wav_5s = wav_5s[:, start:start + 160000]
                        elif N < 160000:
                            wav_5s = F.pad(wav_5s, (0, 160000 - N))
                        perch_emb = perch_teacher.embed(wav_5s).to(device)
                    distill_loss = F.mse_loss(distill_emb, perch_emb)
                    loss = cls_loss + ALPHA_DISTILL * distill_loss
                else:
                    distill_loss = torch.tensor(0.0)
                    loss = cls_loss

            opt.zero_grad()
            scaler.scale(loss).backward()  # 誤差逆伝播: 各パラメータの勾配を計算
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            scaler.step(opt)  # パラメータの更新
            scaler.update()   # スケーラーの状態を更新
            sch.step()
            el += loss.item(); el_cls += cls_loss.item()
            el_dist += distill_loss.item(); nb_count += 1

        # 検証
        val_preds_dict = _predict_from_waveforms(m, mel_transform, val_wavs)
        val_preds = val_preds_dict["blend"]
        r = full_eval(Y_val, val_preds, ns22_val, TAXON_MASKS)
        for mode in ["clip", "fmax", "blend"]:
            r_mode = full_eval(Y_val, val_preds_dict[mode], ns22_val, TAXON_MASKS)
            r[f"ns22_{mode}"] = r_mode["non_s22_macro"]

        history["ep"].append(ep)
        history["train_loss"].append(round(el / nb_count, 5))
        history["cls_loss"].append(round(el_cls / nb_count, 5))
        history["dist_loss"].append(round(el_dist / nb_count, 5))
        history["macro"].append(r["macro_auc_all"])
        history["ns22_macro"].append(r["non_s22_macro"])
        for t in ["Aves", "Amphibia", "Insecta", "Mammalia"]:
            history[f"ns22_{t}"].append(r[f"non_s22_{t}"])
        history["val_preds"].append(val_preds.astype(np.float32))

        tag_ns22 = ""; tag_macro = ""
        if r["non_s22_macro"] > best_ns22:
            best_ns22 = r["non_s22_macro"]
            best_state_ns22 = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_ns22 = " *ns22"
        if r["macro_auc_all"] > best_macro:
            best_macro = r["macro_auc_all"]
            best_state_macro = {k: v.cpu().clone() for k, v in m.state_dict().items()}
            tag_macro = " *macro"

        # --- 変更004 追加 ↓↓↓--------------------------------------------------------------
        # 後半1/3 epochのチェックポイントを保存
        if ep >= int(EPOCHS * 2 / 3):
            soup_states.append({k: v.cpu().clone() for k, v in m.state_dict().items()})
        # --- 変更004 追加 ↑↑↑ -------------------------------------------------------------

        dist_str = f" dist={el_dist/nb_count:.4f}" if USE_PERCH_DISTILL else ""
        print(f"    Ep{ep:02d}: loss={el/nb_count:.4f} cls={el_cls/nb_count:.4f}{dist_str} "
              f"lr={opt.param_groups[0]['lr']:.1e} | "
              f"ns22: {r['ns22_blend']:.4f} | "
              f"Av={r['non_s22_Aves']:.4f} Am={r['non_s22_Amphibia']:.4f} "
              f"In={r['non_s22_Insecta']:.4f} Ma={r['non_s22_Mammalia']:.4f} "
              f"[{time.time()-t0:.0f}s]{tag_ns22}{tag_macro}")

    # --- 変更004 追加 ↓↓↓--------------------------------------------------------------
    # Checkpoint soup — 後半1/3 epochの重みを平均化してbest_macro_stateを上書き
    if len(soup_states) > 1:
        print(f"  Checkpoint soup: {len(soup_states)}個のチェックポイントを平均化")
        avg_state = {}
        for key in soup_states[0].keys():
            avg_state[key] = torch.stack([s[key].float() for s in soup_states]).mean(0)
        best_state_macro = avg_state
        print("  Checkpoint soup 完了")
    # --- 変更004 追加 ↑↑↑ -------------------------------------------------------------

    del perch_teacher, m, mel_transform, spec_augment
    torch.cuda.empty_cache(); gc.collect()
    return best_state_ns22, best_state_macro, history

print("学習関数準備完了")

学習関数準備完了


## S6 — フォールドループ + ONNXエクスポート

1. 各フォールドで以下を順番に実行する。

- A. フォールドループ
  - S5の`train_fold`を全フォールド分繰り返す。
  - 各フォールドのベストチェックポイント（重み）を`.pt`ファイルに保存する。

- B. ONNXエクスポート
  - 保存したベスト重みを`SEDExportWrapper`に読み込み、ONNX形式で書き出す。
  - 出力：`sed_fold0.onnx`〜`sed_fold4.onnx`（フォールド数分）。
  - ※エクスポートするのは分類パス（clipwise・framewise）のみ。蒸留ヘッドは推論に不要なので含めない。

- C. ONNX検証
  - エクスポート直後に、PyTorchの出力とONNXの出力を比較して誤差を確認する。
  - 誤差が1e-1を超えたらエラーにして、変換ミスを検出する。

- ※手法
  - なぜONNXに変換するのか
    - 推論時にPyTorchが不要になる。`onnxruntime` + `librosa`だけで動くため、Kaggleの推論環境（メモリ・時間制限が厳しい）でも高速に動かせる。
  - なぜ蒸留ヘッドを除くのか
    - 蒸留ヘッドは学習時にPerchの表現に近づけるためのもので、推論（予測）には使わない。含めない方がモデルが軽くなり、推論が速くなる。
  - フォールドごとにモデルを保存する理由
    - 各フォールドのモデルを全部残しておくことで、推論時に全フォールドの予測を平均（アンサンブル）できる。1つのモデルより複数の平均の方がスコアのブレが小さくなる。

- ※用語
  - チェックポイント
    - 学習途中・学習完了時のモデルの重みを保存したファイル（`.pt`）。ベストスコアのタイミングで保存する。
  - ONNX（Open Neural Network Exchange）
    - モデルを学習フレームワーク（PyTorch・TensorFlowなど）に依存しない形式で保存する規格。異なる環境でも高速に推論できる。
  - アンサンブル
    - 複数モデルの予測を平均して最終予測にする手法。1モデルより安定したスコアが出る。

In [ ]:
# =================================================================
# S6 -- フォールドループ + ONNXエクスポート
# =================================================================

if MODE != "train":
    print("学習をスキップ（MODE='infer'）")
    oof_ns22 = None
    all_hist = {}
else:

    oof_ns22 = np.full((len(sc_cache_meta), NUM_CLASSES), np.nan, dtype=np.float32)
    all_hist = {}
    for fold_k in FOLDS:
        print(f"\n{'='*60}")
        print(f"FOLD {fold_k}")
        print(f"{'='*60}")
        vm = sc_cache_meta["fold"].values == fold_k
        val_sc_df_k = sc_cache_meta[vm].reset_index(drop=True)

        best_ns22_state, best_macro_state, hist = train_fold(fold_k)
        all_hist[fold_k] = hist

        mel_tf = MelSpecTransform().to(device)
        val_wavs_k = _load_val_waveforms(val_sc_df_k)

        if best_macro_state is not None:
            # PyTorchチェックポイントを保存
            torch.save(best_macro_state, OUT_DIR / f"fold{fold_k}_best_ns22.pt")
            m = make_model()
            m.load_state_dict(best_macro_state, strict=False)
            oof_ns22[vm] = _predict_from_waveforms(m, mel_tf, val_wavs_k)["blend"]

            # --- ONNXエクスポート（安定したトレーシングのためConv1dをリマップ）---
            m.eval()
            INF_N_MELS = N_MELS  # 学習と推論で一致させる
            INF_N_FRAMES = VAL_SAMPLES // HOP_LENGTH + 1

            class SEDExportWrapper(nn.Module):
                def __init__(self, backbone_name, num_classes, backbone_dim, hidden_dim=512):
                    super().__init__()
                    self.backbone = timm.create_model(
                        backbone_name, pretrained=False, in_chans=1,
                        num_classes=0, global_pool="", drop_path_rate=0.15,
                    )
                    self.gem_freq = GeMFreqPool(p_init=3.0)
                    self.dense_drop1 = nn.Dropout(0.25)
                    self.dense_conv = nn.Conv1d(backbone_dim, hidden_dim, 1)
                    self.dense_relu = nn.ReLU(inplace=True)
                    self.dense_drop2 = nn.Dropout(0.5)
                    self.att = nn.Conv1d(hidden_dim, num_classes, 1)
                    self.cla = nn.Conv1d(hidden_dim, num_classes, 1)

                def forward(self, mel):
                    h = self.backbone(mel)
                    h = self.gem_freq(h)
                    h = self.dense_drop1(h)
                    h = self.dense_conv(h)
                    h = self.dense_relu(h)
                    h = self.dense_drop2(h)
                    norm_att = torch.softmax(torch.tanh(self.att(h)), dim=-1)
                    framewise = self.cla(h)
                    clip = torch.sum(norm_att * framewise, dim=2)
                    return clip, framewise.permute(0, 2, 1)

            def load_and_remap_state(export_model, trained_state):
                remap = {}
                for k, v in trained_state.items():
                    if k.startswith("distill_head."):
                        continue
                    if k == "dense.1.weight":
                        remap["dense_conv.weight"] = v.unsqueeze(-1)
                    elif k == "dense.1.bias":
                        remap["dense_conv.bias"] = v
                    else:
                        remap[k] = v
                export_model.load_state_dict(remap, strict=False)

            export_model = SEDExportWrapper(
                BACKBONE_NAME, NUM_CLASSES, m.backbone_dim
            ).to(device)
            load_and_remap_state(export_model, best_macro_state)
            export_model.eval()

            dummy_mel = torch.randn(1, 1, INF_N_MELS, INF_N_FRAMES).to(device)
            onnx_path = OUT_DIR / f"sed_fold{fold_k}.onnx"
            torch.onnx.export(
                export_model, dummy_mel, str(onnx_path),
                input_names=["mel"],
                output_names=["clip_logits", "framewise_logits"],
                dynamic_axes={"mel": {0: "batch"},
                              "clip_logits": {0: "batch"},
                              "framewise_logits": {0: "batch"}},
                opset_version=14,
            )

            # 検証
            _sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
            _onnx_out = _sess.run(None, {'mel': dummy_mel.cpu().numpy()})
            with torch.no_grad():
                _ref_clip, _ref_frame = export_model(dummy_mel)
            _diff = np.abs(_ref_clip.cpu().numpy() - _onnx_out[0]).max()
            print(f"  ONNX verify: max|diff|={_diff:.3e}")
            assert _diff < 1e-1, f"ONNX export diverged: {_diff}"
            del _sess

            size_mb = onnx_path.stat().st_size / 1e6
            print(f"  Exported {onnx_path.name} ({size_mb:.1f} MB)")
            del m, export_model


FOLD 0
  Streams: {'focal': 28906, 'sc': 584, 'pseudo': 47083}  steps/ep: 1196
V2 backbone: (1, 1280, 8, 10)  (C=1280)
Perch ONNX loaded: embed_idx=0

--------------------------------------------------
2026-06-03 23:41:26 EPOCH 0
    Ep00: loss=0.2378 cls=0.2277 dist=0.0101 lr=2.6e-04 | ns22: 0.7781 | Av=0.6540 Am=0.7705 In=0.8074 Ma=0.9583 [1268s] *ns22 *macro

--------------------------------------------------
2026-06-04 00:02:33 EPOCH 1
    Ep01: loss=0.0566 cls=0.0500 dist=0.0066 lr=5.0e-04 | ns22: 0.8568 | Av=0.9904 Am=0.7829 In=0.8422 Ma=1.0000 [1301s] *ns22 *macro

--------------------------------------------------
2026-06-04 00:24:15 EPOCH 2
    Ep02: loss=0.0432 cls=0.0383 dist=0.0049 lr=5.0e-04 | ns22: 0.8965 | Av=0.9855 Am=0.8914 In=0.8623 Ma=1.0000 [1280s] *ns22 *macro

--------------------------------------------------
2026-06-04 00:45:35 EPOCH 3
    Ep03: loss=0.0384 cls=0.0342 dist=0.0042 lr=4.8e-04 | ns22: 0.9039 | Av=0.9851 Am=0.8795 In=0.8879 Ma=1.0000 [1300s] *ns22 

W0603 21:08:10.700000 1132 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


V2 backbone: (1, 1280, 8, 10)  (C=1280)
[torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SEDExportWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:64: adapter_lookup: Assertion `false`

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
  ONNX verify: max|diff|=3.808e-03
  Exported sed_fold0.onnx (0.6 MB)


## S7 — 変更003：少数サンプル対策（12th place解法）

1. S6で学習したモデルに対して、希少種だけを対象に追加学習する。

- A. 少数クラスの特定
  - 20件未満（MINORITY_MIN_SAMPLE）の種を少数クラスとして抽出する。
  - S2のアップサンプリング前の元データを基準にする（複製後だと件数が水増しされているため）。

- B. 少数クラス専用データセット（MinorityFocalDS）
  - 少数クラスの焦点音声だけを使ったデータセットを作る。
  - MixUpなし。MixUpで他の種と混ぜると、少ない教材の特徴が薄まってしまうため。

- C. バックボーンと蒸留ヘッドを凍結
  - `requires_grad = False`でバックボーンと蒸留ヘッドの重みを固定する。
  - SEDヘッド（gem_freq / dense / att / cla）だけを学習対象にする。

- D. 少数クラスで再学習（5エポック）
  - 凍結したモデルのSEDヘッドだけを、少数クラスのデータで追加学習する。
  - 学習率は通常より小さめ（1e-4）。バックボーンが固定されているので大きい学習率は不要。

- E. チェックポイントの上書き
  - 再学習済みの重みでS6のチェックポイントを上書き保存する。
  - 以降のONNXエクスポート・OOF評価は、この再学習済みモデルを使う。

- ※手法
  - なぜバックボーンを凍結するのか
    - 少数クラスのデータは件数が少ないため、バックボーンまで動かすと過学習しやすい。
    - バックボーンはS6で全クラスのデータで十分学習済みなので、特徴抽出はそのまま使える。
    - SEDヘッドだけ動かすことで、「少数クラスの分類の仕方」だけを調整する。
  - なぜMixUpなしにするのか
    - 通常学習ではMixUpで他の種と混ぜて汎化を上げているが、少数クラスは教材が少ない。
    - 混ぜると「その種らしさ」が薄まってしまうため、純粋にその種だけで学ばせる。
  - なぜ5エポックで十分か
    - バックボーンが凍結されていてSEDヘッドだけを動かすので、更新するパラメータが少ない。少ないエポックでもSEDヘッドは十分収束する。多すぎると過学習する。

- ※用語
  - 凍結（freeze）
    - `requires_grad = False`でパラメータの勾配計算をオフにする。学習中に重みが動かなくなる。
  - SEDヘッド
    - GeMFreqPool・ボトルネック・アテンション・分類層の4つ。バックボーンの後ろにあり、特徴から234種の予測を出す部分。
  - 少数クラス（minority class）
    - 学習データが極端に少ない種。モデルが十分学習できず本番でつまずきやすい。

In [ ]:
# --- 変更003 追加 ↓↓↓--------------------------------------------------------------

# =================================================================
# S7 -- 変更003: 少数サンプル対策
# =================================================================

MINORITY_EPOCHS      = 5     # 再学習エポック数（多すぎると過学習）
MINORITY_LR          = 1e-4  # Backbone凍結時は小さめのLRで安定
MINORITY_MIN_SAMPLE  = MIN_SAMPLE  # 通常学習と同じ基準（デフォルト: 20）

# ① 少数クラスのラベルとインデックスを特定
# アップサンプリング前の元データを使用
counts = audio_cache_meta.drop_duplicates("original_idx")["primary_label"].value_counts()
minority_labels  = counts[counts < MINORITY_MIN_SAMPLE].index.tolist()
minority_indices = [LABEL2IDX[lbl] for lbl in minority_labels if lbl in LABEL2IDX]
print(f"少数クラス数: {len(minority_indices)} / {NUM_CLASSES}")

# ② 少数クラスの学習データのみ抽出
minority_meta = audio_cache_meta[
    audio_cache_meta["primary_label"].isin(minority_labels)
].reset_index(drop=True)
print(f"少数クラスの学習サンプル数: {len(minority_meta)}")

# ③ 少数クラス専用データセット
#    通常のFocalDSと異なりMixUpなし（少数クラスの特徴を薄めないため）
class MinorityFocalDS(Dataset):
    def __init__(self, meta, aug=True):
        self.meta = meta
        self.aug  = aug

    def __len__(self):
        return len(self.meta)

    def __getitem__(self, idx):
        row = self.meta.iloc[idx]
        wav = load_focal(row["cache_file"])
        if wav is None:
            wav = np.zeros(TRAIN_SAMPLES, dtype=np.float32)
        wav = extract_chunk_np(wav, 0, TRAIN_SAMPLES).astype(np.float32)
        if self.aug:
            wav = apply_aug(wav)
        lb = np.zeros(NUM_CLASSES, dtype=np.float32)
        for lbl in str(row["primary_label"]).split(";"):
            if lbl in LABEL2IDX:
                lb[LABEL2IDX[lbl]] = 1.0
        return torch.from_numpy(wav).unsqueeze(0), torch.from_numpy(lb)

# ④ 各foldのモデルに対して少数クラス再学習を実施
for fold_k in FOLDS:
    ckpt_path = OUT_DIR / f"fold{fold_k}_best_ns22.pt"
    if not ckpt_path.exists():
        print(f"fold{fold_k}: チェックポイントが見つかりません → スキップ")
        continue

    print(f"\n{'='*60}")
    print(f"少数サンプル再学習 FOLD {fold_k}")
    print(f"{'='*60}")

    # 通常学習済みモデルを読み込む
    m = BirdSEDModel().to(device)
    m.load_state_dict(torch.load(ckpt_path, map_location=device), strict=False)

    # ⑤ Backboneと蒸留ヘッドを凍結
    #    → SEDヘッド（gem_freq / dense / att / cla）のみ学習対象
    for param in m.backbone.parameters():
        param.requires_grad = False
    if hasattr(m, "distill_head"):
        for param in m.distill_head.parameters():
            param.requires_grad = False

    trainable_params = [p for p in m.parameters() if p.requires_grad]
    print(f"学習可能パラメータ数: {sum(p.numel() for p in trainable_params):,}")

    opt    = torch.optim.AdamW(trainable_params, lr=MINORITY_LR, weight_decay=WD)
    scaler = GradScaler()

    mel_transform = MelSpecTransform().to(device)
    spec_augment  = SpecAugment().to(device)

    ds = MinorityFocalDS(minority_meta, aug=True)
    tl = DataLoader(ds, batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best_state = None
    best_loss  = float("inf")

    for ep in range(MINORITY_EPOCHS):
        m.train()
        el, nb_count = 0.0, 0

        for wav, lb in tl:
            wav, lb = wav.to(device), lb.to(device)

            # メルスペクトログラム変換（勾配不要）
            with torch.no_grad():
                mel = mel_transform(wav)
                B = mel.size(0)
                for i in range(B):
                    mel[i] = (mel[i] - mel[i].mean()) / (mel[i].std() + 1e-6)
                mel = spec_augment(mel)

            opt.zero_grad()
            with autocast():
                clip_logits, framewise = m(mel, return_framewise=True)
                frame_max = framewise.max(dim=1).values
                # clip と frame_max を等重みでBCEを計算
                bce_clip  = F.binary_cross_entropy_with_logits(clip_logits, lb, reduction="mean")
                bce_frame = F.binary_cross_entropy_with_logits(frame_max,   lb, reduction="mean")
                loss      = 0.5 * bce_clip + 0.5 * bce_frame

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(opt)
            scaler.update()
            el += loss.item(); nb_count += 1

        avg_loss = el / max(nb_count, 1)
        print(f"  Ep{ep:02d}: loss={avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss  = avg_loss
            best_state = {k: v.cpu().clone() for k, v in m.state_dict().items()}

    # ⑥ 再学習済みモデルで元のチェックポイントを上書き保存
    #    → S7（OOF評価）・ONNX出力が再学習済みモデルを使用する
    if best_state is not None:
        torch.save(best_state, ckpt_path)
        print(f"  上書き保存: {ckpt_path}")

    del m, opt, scaler, mel_transform, spec_augment
    torch.cuda.empty_cache()

print("\n少数サンプル再学習 完了")

# --- 変更003 追加 ↑↑↑ -------------------------------------------------------------

少数クラス数: 36 / 234
少数クラスの学習サンプル数: 838

少数サンプル再学習 FOLD 0
V2 backbone: (1, 1280, 8, 10)  (C=1280)
学習可能パラメータ数: 895,957
  Ep00: loss=0.0170
  Ep01: loss=0.0135
  Ep02: loss=0.0114
  Ep03: loss=0.0099
  Ep04: loss=0.0087
  上書き保存: /content/drive/MyDrive/private/Kaggle/BirdCLEF_2026/Your_Work/training_sed2/output_003_004_009/fold01234_epoch25/fold0_best_ns22.pt

少数サンプル再学習 完了


## S8 — OOF評価

アウトオブフォールド性能の簡易サマリー。


In [ ]:
# =================================================================
# S8 -- OOF評価
# =================================================================

if MODE != "train":
    print("評価をスキップ（MODE='infer'）")
else:

    has = ~np.isnan(oof_ns22[:, 0])
    if has.sum() > 0:
        r_all = full_eval(Y_SC[has], oof_ns22[has], non_s22_mask_sc[has], TAXON_MASKS)
        print("=" * 60)
        print("OOF RESULTS (best-ns22 checkpoints)")
        print("=" * 60)
        print(f"  macro AUC (all):        {r_all['macro_auc_all']:.4f}")
        print(f"  macro AUC (non-S22):    {r_all['non_s22_macro']:.4f}")
        for t in ["Aves", "Amphibia", "Insecta", "Mammalia"]:
            print(f"    {t:<12}: {r_all.get(f'non_s22_{t}', float('nan')):.4f}")

    # エポックごとの推移
    print("\nPer-epoch pooled non-S22 AUC:")
    fold_true, fold_ns22_m = {}, {}
    for fk in range(N_FOLDS):
        vm = sc_cache_meta["fold"].values == fk
        fold_true[fk] = Y_SC[vm]
        fold_ns22_m[fk] = non_s22_mask_sc[vm]

    n_eps = [len(all_hist[k]["val_preds"]) for k in range(N_FOLDS) if k in all_hist]
    max_ep = min(n_eps) if n_eps else 0
    for ep in range(max_ep):
        pp = np.concatenate([all_hist[k]["val_preds"][ep] for k in range(N_FOLDS) if k in all_hist])
        pt = np.concatenate([fold_true[k] for k in range(N_FOLDS) if k in all_hist])
        pm = np.concatenate([fold_ns22_m[k] for k in range(N_FOLDS) if k in all_hist])
        ns, _ = compute_macro_auc(pt, pp, mask=pm)
        print(f"  Ep{ep:02d}: {ns:.4f}")

OOF RESULTS (best-ns22 checkpoints)
  macro AUC (all):        0.8901
  macro AUC (non-S22):    0.8556
    Aves        : 0.9922
    Amphibia    : 0.8920
    Insecta     : 0.7693
    Mammalia    : 1.0000

Per-epoch pooled non-S22 AUC:
  Ep00: 0.7781
  Ep01: 0.8568
  Ep02: 0.8965
  Ep03: 0.9039
  Ep04: 0.9029
  Ep05: 0.9022
  Ep06: 0.8913
  Ep07: 0.8883
  Ep08: 0.8763
  Ep09: 0.8692
  Ep10: 0.8669
  Ep11: 0.8594
  Ep12: 0.8554
  Ep13: 0.8587
  Ep14: 0.8545
  Ep15: 0.8545
  Ep16: 0.8558
  Ep17: 0.8571


## S9 — セッションの終了

In [ ]:
# =================================================================
# S9 -- セッションの終了
# =================================================================

import os, subprocess
from google.colab import runtime

# セッションを終了
session_id = os.environ.get('COLAB_JUPYTER_TOKEN', '')
subprocess.run(['curl', '-X', 'DELETE',
    f'http://localhost:9000/api/sessions/{session_id}'])

# ランタイムを終了（CUの消費を止める）
runtime.unassign()